In [ ]:
from src.preprocess.dataset_custom import NewsPaperData, CustomDataset
from src.classifier.classifier import ClassifierModels
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


import os
import pandas as pd
from typing import List, Dict, Tuple
import numpy as np

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import silhouette_score
from sklearn.metrics import classification_report, make_scorer, accuracy_score, precision_score, recall_score, f1_score

# Lendo dados

In [ ]:
def read_data():
  # ------- Instance Train and Test --------- #
  
  dataset_train = NewsPaperData(
      "/src/dataset/abp/",
      "abp_valid.csv"
  )

  dataset_test = NewsPaperData(
      "/src/dataset/abp/",
      "abp_random_test.csv"
  )

  # ------- Train ---------- #
  df_train, _ = dataset_train.get_dataset_custom(
      dict_rename={"bias": "labels", "content": "text"}
  )

  df_test, _ = dataset_test.get_dataset_custom(
      dict_rename={"bias": "labels", "content": "text"}
  )

  df_train = df_train.dropna(subset=["text"]).reset_index(drop=True)
  df_test = df_test.dropna(subset=["text"]).reset_index(drop=True)

  return df_train, df_test

# Gerando embeddings

In [ ]:
from src.networks.networks import TransformerNet

In [ ]:
def extract_features_text(model, ds_loader, shape_feature, device):
  with torch.no_grad():
      model.eval()
      embeddings = np.zeros((len(ds_loader.dataset), shape_feature)) # numero do max_len
      labels = np.zeros(len(ds_loader.dataset))
      k = 0
      for sample, targets in ds_loader:
          inputs = (
              sample["ids"].to(device=device),
              sample["mask"].to(device=device)
          )
          # o 64 é o tamanho do batch
          #x = model.get_embeddings(inputs[0], inputs[1]).data.cpu().numpy()
          embeddings[k:k+4] = model.get_embeddings(inputs[0], inputs[1]).data.cpu().numpy()
          labels[k:k+4] = targets.numpy()
          k += 4
  return embeddings, labels

# Carregando Modelo Gerador de Embeddings

In [ ]:
from transformers import AutoTokenizer

In [ ]:
def get_model(path_model, tokenizer_name):
  tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_name
  )

  device = "cuda" if torch.cuda.is_available() else "cpu"

  model = TransformerNet(name_model=tokenizer_name).to(device)

  # 2. Carregue o dicionário de pesos (o seu OrderedDict)
  state_dict = torch.load(path_model, map_location=device)

  # 3. Carregue os pesos para dentro da instância do modelo
  model.load_state_dict(state_dict, strict=False)

  return model, tokenizer

In [ ]:
model, tokenizer = get_model(
    '/src/outputs/contrastive_droberta_flip/{name_model_saved}.pth',
    tokenizador # - 'distilbert/distilroberta-base'
)

# Avaliação

In [ ]:
test_parameters = {
  "batch_size": 4,
  "shuffle": True,
  "num_workers": 0
}

df_valid, df_test = read_data()

ds_abp_test = CustomDataset(
    df=df_test,
    tokenizer=tokenizer,
    max_len=512
)

test_idx_loader_abp = DataLoader(
    ds_abp_test,
    **test_parameters
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
emb_test, labels_test = extract_features_text(
    model, 
    test_idx_loader_abp, 
    768, 
    device
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    emb_test, labels_test, test_size=0.25, random_state=42
)

In [ ]:
clf_knn = ClassifierModels("knn").get_model_clf()

clf_knn.fit(X_train, y_train)

best_knn = clf_knn.best_estimator_

y_pred = best_knn.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Left', 'Center', 'Right']))

In [ ]:
# MLP com Kfold
best_acc = 0
best_mlp = None

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

    # Definir a rede neural
    mlp = ClassifierModels("mlp").get_model_clf()

    # Treinar
    mlp.fit(X_fold_train, y_fold_train)

    # Validar
    val_preds = mlp.predict(X_fold_val)
    val_acc = accuracy_score(y_fold_val, val_preds)

    print(f"Fold {fold+1} Val Acc: {val_acc:.4f}")

    # Lógica para guardar o melhor
    if val_acc > best_acc:
        best_acc = val_acc
        best_mlp = mlp
        print(f"⭐ Novo melhor modelo encontrado no Fold {fold+1}!")


test_preds = best_mlp.predict(X_test)
print(classification_report(y_test, test_preds))


In [ ]:
# classificador com o kmeans
n_clusters = 3
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_kmeans = None
best_inertia = float('inf')

# 2. Loop de K-Fold (Treino e Validação)
# Nota: K-Means não usa 'y' no fit, mas usamos y_val para monitorar o alinhamento
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_train_fold = X_train[train_idx]
    X_val_fold = X_train[val_idx]

    # Treinando o modelo no fold de treino
    kmeans = ClassifierModels("kmeans").get_model_clf()
    kmeans.fit(X_train_fold)

    # Validação pela Inércia (métrica interna do K-Means)
    current_inertia = kmeans.inertia_

    print(f"Fold {fold+1} | Inércia: {current_inertia:.2f}")

    # Salvando o melhor modelo baseado na menor inércia
    if current_inertia < best_inertia:
        best_inertia = current_inertia
        best_kmeans = kmeans
        print(f"  ⭐ Novo melhor modelo encontrado!")

# ---------------------------------------------------------
# 3. Teste Final e Cálculo de Métricas (Acc, Precision, Recall, F1)
# ---------------------------------------------------------

# Função para mapear clusters (0, 1, 2) para suas labels reais
def get_mapped_labels(clusters, y_true):
    mapped_labels = np.zeros_like(clusters)
    for i in range(n_clusters):
        mask = (clusters == i)
        if np.any(mask):
            # Atribui ao cluster a label real que mais aparece nele
            real_label = mode(y_true[mask], keepdims=True)[0]
            mapped_labels[mask] = real_label
    return mapped_labels

# Predição no conjunto de teste externo
test_clusters = best_kmeans.predict(X_test)
y_pred_mapped = get_mapped_labels(test_clusters, y_test)

# Exibição dos resultados
print("\n" + "="*30)
print("RESULTADOS FINAIS - K-MEANS")
print("="*30)
print(classification_report(y_test, y_pred_mapped,
                            target_names=['Classe 0', 'Classe 1', 'Classe 2']))

# Métrica extra de qualidade de cluster
sil = silhouette_score(X_test, test_clusters)
print(f"Acurácia Global: {accuracy_score(y_test, y_pred_mapped):.4f}")
print(f"Coeficiente de Silhueta: {sil:.4f}")